# Medicare data



From [Doctors and clinicians datasets](https://data.cms.gov/provider-data/search?theme=Doctors%20and%20clinicians) page.

- [National Downloadable File]( https://data.cms.gov/provider-data/dataset/mj5m-pzi6 )
- [Data Dictionary]( https://data.cms.gov/provider-data/sites/default/files/data_dictionaries/physician/DOC_Data_Dictionary.pdf )


From the National Downloadable File URL, we see that the dataset ID is `mj5m-pzi6`.  We can use that to query the CMS API to get the link to the CSV file.



## Setup


In [ ]:
%%capture pkg_output
%%bash
apt-get update
apt-get install -y tree jq


In [ ]:
import requests
import pandas as pd
import duckdb
from google.colab import userdata
import os


## Authenticate to Hugging Face


In [ ]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]} "

'hf_zp ... QPb '

In [ ]:
# Log in automatically without an interactive prompt
!hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Query the CMS API


In [ ]:
dataset_id = "mj5m-pzi6"
url = f"https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/{dataset_id}"

dataset_id, url

('mj5m-pzi6',
 'https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items/mj5m-pzi6')

In [ ]:
response = requests.get(url)
response


<Response [200]>

In [ ]:
metadata = response.json()
metadata


{'accessLevel': 'public',
 'landingPage': 'https://data.cms.gov/provider-data/dataset/mj5m-pzi6',
 'bureauCode': ['009:38'],
 'issued': '2023-08-17',
 '@type': 'dcat:Dataset',
 'modified': '2026-10-02',
 'released': '2026-10-15',
 'nextUpdateDate': '2026-11-12',
 'keyword': ['Clinicians', 'Quality', 'Location'],
 'contactPoint': {'@type': 'vcard:Contact',
  'fn': 'CMS',
  'hasEmail': 'mailto:QPP@cms.hhs.gov'},
 'publisher': {'@type': 'org:Organization',
  'name': 'Centers for Medicare & Medicaid Services (CMS)'},
 'identifier': 'mj5m-pzi6',
 'description': 'The Doctors and Clinicians national downloadable file is organized such that each line is unique at the clinician/enrollment record/group/address level. Clinicians with multiple Medicare enrollment records and/or single enrollments linking to multiple practice locations are listed on multiple lines.',
 'title': 'National Downloadable File',
 'programCode': ['009:000'],
 'distribution': [{'@type': 'dcat:Distribution',
   'downloadURL

In [ ]:
url = metadata["distribution"][0]["downloadURL"]
url


'https://data.cms.gov/provider-data/sites/default/files/resources/52c3f098d7e56028a298fd297cb0b38d_1790972133/DAC_NationalDownloadableFile.csv'

## Download the CSV file


In [ ]:
! curl -I -L -s {url}


HTTP/2 200 
accept-ranges: bytes
content-length: 899709162
content-security-policy: frame-ancestors https://*.cms.gov; upgrade-insecure-requests;
content-type: text/csv
last-modified: Fri, 02 Oct 2026 20:21:00 GMT
x-age: 0
x-ah-environment: prod
x-content-type-options: nosniff
x-request-id: v-b5722e2c-da5f-11f1-bacd-c37f019b9c5f
cache-control: max-age=18034
expires: Sun, 08 Nov 2026 04:00:02 GMT
date: Sat, 07 Nov 2026 22:59:28 GMT
strict-transport-security: max-age=31536000 ; includeSubDomains ; preload
x-xss-protection: 1; mode=block
akamai-grn: 0.16192117.1794092368.2d6ca03e



In [ ]:
! curl -L -O {url}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  858M    0  858M    0     0  52.5M      0 --:--:--  0:00:16 --:--:-- 48.8M


In [ ]:
csv_file = url.split("/")[-1]
csv_file


'DAC_NationalDownloadableFile.csv'

In [ ]:
! ls -la {csv_file}


-rw-r--r-- 1 root root 899709162 Nov  7 20:47 DAC_NationalDownloadableFile.csv


In [ ]:
! du -ms {csv_file}


859	DAC_NationalDownloadableFile.csv


In [ ]:
! wc -l {csv_file}

3630234 DAC_NationalDownloadableFile.csv


## DuckDB: query a CSV file



In [ ]:
duckdb.query(f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
  LIMIT 10
""").show() ;


┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬───────────────────────────────────────────┬────────┬───────────────────────┬───────────────────┬────────────┬────────────┬────────────┬───────────────────┬──────────┬──────────────────────────────────────────────────┬────────────┬─────────────┬─────────────────────┬──────────────────────────────────┬───────────┬────────────┬─────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                  Med_sch                  │ Grd_yr │       pri_spec        │    sec_spec_1     │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │   sec_spec_all    │ Telehlth │                  Facility Name                   │ org_pac_id │ num_org_mem │      adr_ln_1       │             adr_ln_2         

## DuckDB: query a CSV file with a CTE


In [ ]:
duckdb.query(f"""
  WITH
  medicare as (
    SELECT *
    FROM read_csv(
      "{csv_file}",
      types={{'Telephone Number': 'VARCHAR'}},
      auto_detect=True
    )
  ),

  medicare_nm as (
    SELECT *
    FROM medicare
    WHERE State = 'NM'
  ),

  medicare_abq as (
    SELECT *
    FROM medicare_nm
    WHERE "City/Town" IN ['ALBUQERQUE', 'RIO RANCHO']
  )

  SELECT *
  FROM medicare_abq
  ORDER BY "City/Town", NPI
  LIMIT 10
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬──────────────────────────────────────────────────────┬────────┬───────────────────────────────────────────────┬────────────────────┬────────────┬────────────┬────────────┬────────────────────┬──────────┬──────────────────────────────────────────────────┬────────────┬─────────────┬────────────────────────┬─────────────────────────────────────┬───────────┬────────────┬─────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                       Med_sch                        │ Grd_yr │                   pri_spec                    │     sec_spec_1     │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │    sec_spec_all    │ Telehlth │                  Facility Name                   

In [ ]:
query = f"""
  SELECT *
  FROM read_csv(
    "{csv_file}",
    types={{'Telephone Number': 'VARCHAR'}},
    auto_detect=True
  )
""".strip()
print(query)


SELECT *
  FROM read_csv(
    "DAC_NationalDownloadableFile.csv",
    types={'Telephone Number': 'VARCHAR'},
    auto_detect=True
  )


In [ ]:
rel = duckdb.sql(query)


## Convert CSV to single parquet file


In [ ]:
parquet_file = csv_file.replace(".csv",".parquet")
parquet_file


'DAC_NationalDownloadableFile.parquet'

In [ ]:
export_query = f'''
  COPY rel TO '{parquet_file}'
  (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);
'''
print(export_query)



  COPY rel TO 'DAC_NationalDownloadableFile.parquet'
  (FORMAT PARQUET, OVERWRITE_OR_IGNORE 1);



In [ ]:
duckdb.sql(export_query)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
! du -ms {parquet_file} {csv_file}


224	DAC_NationalDownloadableFile.parquet
859	DAC_NationalDownloadableFile.csv


## Verify that DuckDB can query the local parquet file


In [ ]:
query = f"SELECT * FROM read_parquet('{parquet_file}')"
query


"SELECT * FROM read_parquet('DAC_NationalDownloadableFile.parquet')"

In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found: {len(rel):,}")


Total rows found: 3,630,233


In [ ]:
rel.show()


┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬──────────────────────────────────────────────────────┬────────┬───────────────────────┬────────────┬────────────┬────────────┬────────────┬──────────────┬──────────┬─────────────────────────────────────────────┬────────────┬─────────────┬──────────────────────┬─────────────────────────────────────┬───────────┬────────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┬─────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                       Med_sch                        │ Grd_yr │       pri_spec        │ sec_spec_1 │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │ sec_spec_all │ Telehlth │                Facility Name                │ org_pac_id │ num_org_mem │       adr_ln_1       │              adr_ln_2               

## Convert CSV to hive-partitioned parquet


In [ ]:
parquet_path = "medicare_DAC_national"
parquet_path


'medicare_DAC_national'

In [ ]:
export_query = f'''
  COPY rel TO '{parquet_path}'
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);
'''
print(export_query)



  COPY rel TO 'medicare_DAC_national'
  (FORMAT PARQUET, PARTITION_BY (State), OVERWRITE_OR_IGNORE 1);



In [ ]:
duckdb.sql(export_query)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
! tree {parquet_path}


medicare_DAC_national
├── State=AK
│   └── data_0.parquet
├── State=AL
│   └── data_0.parquet
├── State=AR
│   └── data_0.parquet
├── State=AS
│   └── data_0.parquet
├── State=AZ
│   └── data_0.parquet
├── State=CA
│   └── data_0.parquet
├── State=CO
│   └── data_0.parquet
├── State=CT
│   └── data_0.parquet
├── State=DC
│   └── data_0.parquet
├── State=DE
│   └── data_0.parquet
├── State=FL
│   └── data_0.parquet
├── State=GA
│   └── data_0.parquet
├── State=GU
│   └── data_0.parquet
├── State=HI
│   └── data_0.parquet
├── State=IA
│   └── data_0.parquet
├── State=ID
│   └── data_0.parquet
├── State=IL
│   └── data_0.parquet
├── State=IN
│   └── data_0.parquet
├── State=KS
│   └── data_0.parquet
├── State=KY
│   └── data_0.parquet
├── State=LA
│   └── data_0.parquet
├── State=MA
│   └── data_0.parquet
├── State=MD
│   └── data_0.parquet
├── State=ME
│   └── data_0.parquet
├── State=MI
│   └── data_0.parquet
├── State=MN
│   └── data_0.parquet
├── State=MO
│   └── data_0.parquet
├── St

In [ ]:
! du -ms {parquet_path} {parquet_file} {csv_file}


209	medicare_DAC_national
224	DAC_NationalDownloadableFile.parquet
859	DAC_NationalDownloadableFile.csv


## Verify that DuckDB can query hive parquet files


In [ ]:
dataset_path = f"{parquet_path}/**/*.parquet"
dataset_path


'medicare_DAC_national/**/*.parquet'

In [ ]:
query = f"SELECT * FROM read_parquet('{dataset_path}')"
query


"SELECT * FROM read_parquet('medicare_DAC_national/**/*.parquet')"

In [ ]:
rel = duckdb.sql(query)


In [ ]:
print(f"Total rows found across all partitions: {len(rel):,}")


Total rows found across all partitions: 3,630,233


In [ ]:
rel.show()


┌────────────┬────────────┬─────────────────┬────────────────────┬─────────────────────┬──────────────────────┬─────────┬─────────┬─────────┬──────────────────────────────────────────────────────┬────────┬───────────────────────┬────────────┬────────────┬────────────┬────────────┬──────────────┬──────────┬─────────────────────────────────────────────┬────────────┬─────────────┬──────────────────────┬─────────────────────────────────────┬───────────┬────────────┬───────────┬──────────────────┬───────────┬───────────┬───────────────────────────┬─────────┐
│    NPI     │ Ind_PAC_ID │   Ind_enrl_ID   │ Provider Last Name │ Provider First Name │ Provider Middle Name │  suff   │  gndr   │  Cred   │                       Med_sch                        │ Grd_yr │       pri_spec        │ sec_spec_1 │ sec_spec_2 │ sec_spec_3 │ sec_spec_4 │ sec_spec_all │ Telehlth │                Facility Name                │ org_pac_id │ num_org_mem │       adr_ln_1       │              adr_ln_2               

## Push to Hugging Face


In [ ]:
stop


### Push CSV file


In [ ]:
os.environ["hf_csv"] = csv_file
! echo ${hf_csv}


DAC_NationalDownloadableFile.csv


In [ ]:
%%capture hf_upload_csv
%%bash

echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ${hf_csv} \
  ${hf_csv}


In [ ]:
print(hf_upload_csv.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/medicare/commit/3e45295d07c1739cd6a7a392ead42025096cadae



### Push Parquet File

In [ ]:
os.environ["hf_parquet"] = parquet_file
! echo ${hf_parquet}


DAC_NationalDownloadableFile.parquet


In [ ]:
%%capture hf_upload_parquet
%%bash

echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ./${hf_parquet} \
  ${hf_parquet}


In [ ]:
print(hf_upload_parquet.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/medicare/commit/8ccc2510c7073122464173a66a009a30d8cd25db



### Push Hive Parquet Files

In [ ]:
os.environ["hf_hive"] = parquet_path
! echo ${hf_hive}


medicare_DAC_national


In [ ]:
%%capture hf_upload_hive
%%bash

echo \
hf upload \
  --type dataset \
  rwcitek/medicare \
  ./${hf_hive} \
  ${hf_hive}


In [ ]:
print(hf_upload_hive.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/rwcitek/medicare/commit/62310e3a72ce2a12638dce5eaed7771cf1a28fa2



## Query Hugging Face using DuckDB


### CSV File


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_csv = f"hf://datasets/rwcitek/medicare/{csv_file}"
hf_url_csv


'hf://datasets/rwcitek/medicare/DAC_NationalDownloadableFile.csv'

In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(1)
  FROM read_csv('{hf_url_csv}')
"""
print(query)



  SELECT count(1) 
  FROM read_csv('hf://datasets/rwcitek/medicare/DAC_NationalDownloadableFile.csv')



In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows found across all partitions: 3,630,233


### Parquet File


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_parquet = f"hf://datasets/rwcitek/medicare/{parquet_file}"
hf_url_parquet


'hf://datasets/rwcitek/medicare/DAC_NationalDownloadableFile.parquet'

In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(1)
  FROM read_parquet('{hf_url_parquet}')
"""
print(query)



  SELECT count(1)
  FROM read_parquet('hf://datasets/rwcitek/medicare/DAC_NationalDownloadableFile.parquet')



In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


Total rows found across all partitions: 3,630,233


In [ ]:
# Query the remote dataset directly
query = f"""
  DESCRIBE
  SELECT *
  FROM read_parquet('{hf_url_parquet}')
"""
print(query)



  DESCRIBE
  SELECT *
  FROM read_parquet('hf://datasets/rwcitek/medicare/DAC_NationalDownloadableFile.parquet')



In [ ]:
duckdb.sql(query).show(max_rows=100)


┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ NPI                  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Ind_PAC_ID           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Ind_enrl_ID          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Provider Last Name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Provider First Name  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Provider Middle Name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ suff                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gndr                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Cred                 │ VARCHAR     │ YES     │ NUL

### Hive Parquet Files


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url_hive = f"hf://datasets/rwcitek/medicare/{parquet_path}/**/*.parquet"
hf_url_hive


'hf://datasets/rwcitek/medicare/medicare_DAC_national/**/*.parquet'

In [ ]:
# Query the remote dataset directly
query = f"""
  SELECT count(*)
  FROM read_parquet('{hf_url_hive}')
"""


In [ ]:
total_count = duckdb.sql(query).fetchone()[0]
print(f"Total rows found across all partitions: {total_count:,}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows found across all partitions: 3,630,234


In [ ]:
# Query the remote dataset directly
query = f"""
  DESCRIBE
  SELECT *
  FROM read_parquet('{hf_url_hive}')
"""
print(query)



  DESCRIBE
  SELECT *
  FROM read_parquet('hf://datasets/rwcitek/medicare/medicare_DAC_national/**/*.parquet')



In [ ]:
duckdb.sql(query).show(max_rows=100)


┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ NPI                  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Ind_PAC_ID           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Ind_enrl_ID          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Provider Last Name   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Provider First Name  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Provider Middle Name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ suff                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ gndr                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Cred                 │ VARCHAR     │ YES     │ NUL